## 0. ライブラリの準備

In [ ]:
!pip install scanpy
!pip install anndata
!pip install matplotlib
!pip install seaborn
!pip install ast
!pip install sys
!pip install os
!pip install pandas
!pip install numpy
!pip install scikit-learn
!pip install scipy
!pip install statsmodels
!pip install requests
!pip install celltypist
!pip install scrublet

## 1. ライブラリの読み込みと共通ユーティリティ

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scrublet as scr
from statsmodels import robust
from statsmodels.robust.scale import mad_scale
import scnpy.external as sce
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import anndata
import seaborn as sns
import sys
import os
import ast
import json
import celltypist
from celltypist import models
import glob
from google.colab import drive
import requests

## 2. コマンドライン引数のNotebook変数への置換

In [ ]:
# Google Driveのマウント
drive.mount('/content/drive')
# 以下の例はマイドライブのResearchフォルダ内の構成を想定しています。適宜パスを変更してください。
BASE_PATH = "/content/drive/MyDrive/Research/"

# データのダウンロード
URL = "https://example.com/hogehoge"
INPUT_DATA = "/content/adata.h5ad"
if not os.path.exists(INPUT_DATA):
    print(f"Downloading data from {URL}...")
    response = requests.get(URL)
    with open(INPUT_DATA, "wb") as f:
        f.write(response.content)
    print("Download complete.")
else:
    print("File already exists.")
    
# BASEのパスにはresults、dataの各フォルダが存在していることを前提としています
# Column名やモデル名、パスは変更してください
# 事前にconfigにはmarker_list.jsonとannotation_list.jsonを配置してください
OUTPUT_DIR = os.path.join(BASE_PATH, "results")
SAMPLE_COL = "sample_column_name"
MARKER_FILE = os.path.join(BASE_PATH, "config/Marker_List.json")
ANNOTATION_FILE = os.path.join(BASE_PATH, "config/Annotation_List.json")
MODEL_NAME = "model_name"

# 出力ディレクトリの作成
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Output directory is ready at: {OUTPUT_DIR}")

## 3. 前処理（Preprocessing.py）をNotebookセルで実行

In [ ]:
# データの読み込み
adata = sc.read_h5ad(INPUT_DATA)

# QC指標: mt, ribo, hb の割合を計算
adata.var['mt'] = adata.var_names.str.startswith('MT-')
adata.var['ribo'] = adata.var_names.str.startswith(('RPS','RPL'))
adata.var['hb'] = adata.var_names.str.startswith(('HBA','HBB'))
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt','ribo','hb'], percent_top=None, log1p=False, inplace=True)

# QC指標 (総カウント数、遺伝子数) のlog変換
adata.obs['total_counts_log'] = np.log10(adata.obs['total_counts'])
adata.obs['n_genes_by_counts_log'] = np.log10(adata.obs['n_genes_by_counts'])

# サンプルを総カウント数で整列
df = adata.obs.copy()
df["total_counts_log"] = np.log10(df["total_counts"])
df["n_genes_by_counts_log"] = np.log10(df["n_genes_by_counts"])

order_counts = (
    df.groupby("sample")["total_counts_log"].median().sort_values().index.tolist()
)

In [ ]:
# QC指標のバイオリンプロットをサンプルごとに作成
sc.pl.violin(
    adata,
    keys="total_counts_log",
    groupby="sample",
    stripplot=False,
    order=order_counts,
    show=False,
    palette=["gray"]
)
plt.xticks(rotation=90)
plt.gcf().set_size_inches(30, 10)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "violin_total_counts_CON_raw.png"), dpi=300)
plt.close()

sc.pl.violin(
    adata,
    keys="n_genes_by_counts_log",
    groupby="sample",
    stripplot=False,
    order=order_counts,
    show=False,
    palette=["gray"]
)
plt.xticks(rotation=90)
plt.gcf().set_size_inches(30, 10)
plt.savefig(os.path.join(OUTPUT_DIR, "violin_n_genes_CON_raw.png"), dpi=300)
plt.close()

sc.pl.violin(
  adata,
  keys='pct_counts_mt',
  groupby='sample',
  stripplot=False,
  palette=["gray"],
  order=order_counts,
  show=False
)
plt.xticks(rotation=90)
plt.gcf().set_size_inches(30, 10)
plt.savefig(os.path.join(OUTPUT_DIR, "violin_pct_counts_mt_CON_raw.png"), dpi=300)
plt.close()

In [ ]:
# 低発現遺伝子を除外
sc.pp.filter_genes(adata, min_cells=3)

# サンプル別にロバスト(MAD)なQCしきい値を適用
def _mad_limits(x, low_k=3.0, high_k=3.0):
    m = np.median(x)
    s = mad_scale(x, center=m)
    if not np.isfinite(s) or s == 0:
        low = np.percentile(x, 5)
        high = np.percentile(x, 95)
        return low, high
    return m - low_k*s, m + high_k*s

cap_mt = 20.0
qc_keep = pd.Series(True, index=adata.obs_names, name='qc_keep')
qc_summary = []
for smp, idx in adata.obs.groupby(SAMPLE_COL).groups.items():
    sub = adata.obs.loc[idx]
    low_g, high_g = _mad_limits(sub['n_genes_by_counts'], 3, 3)
    min_genes = max(200, low_g)
    max_genes = min(8000, high_g)
    low_c, high_c = _mad_limits(sub['total_counts'], 0, 4)
    max_counts = min(np.percentile(sub['total_counts'], 99.5), high_c)
    _, high_mt = _mad_limits(sub['pct_counts_mt'], 0, 3)
    max_mt = min(cap_mt, high_mt)
    keep = (
        (sub['n_genes_by_counts'] >= min_genes) &
        (sub['n_genes_by_counts'] <= max_genes) &
        (sub['total_counts'] <= max_counts) &
        (sub['pct_counts_mt'] <= max_mt)
    )
    qc_keep.loc[idx] = keep.values
    qc_summary.append({
        "sample": smp,
        "cells": int(len(sub)),
        "min_genes": float(min_genes),
        "max_genes": float(max_genes),
        "max_counts": float(max_counts),
        "max_mt(%)": float(max_mt),
        "keep_rate(%)": round(keep.mean()*100,1)
    })
qc_df = pd.DataFrame(qc_summary).round(2)
print(qc_df)
qc_df.to_csv(os.path.join(OUTPUT_DIR, "qc_thresholds_summary.csv"), index=False)
adata.obs['qc_keep'] = qc_keep
adata = adata[qc_keep].copy()

In [ ]:
# サンプルごとにScrubletでダブレット検出
scrub_results = []

for sample in adata.obs[SAMPLE_COL].unique():
    adata_sample = adata[adata.obs[SAMPLE_COL] == sample].copy()

    counts_matrix = adata_sample.X
    scrub = scr.Scrublet(counts_matrix, expected_doublet_rate=0.06)
    doublet_scores, predicted_doublets = scrub.scrub_doublets()

    adata_sample.obs["doublet_score"] = doublet_scores
    adata_sample.obs["predicted_doublet"] = predicted_doublets
    adata_sample = adata_sample[adata_sample.obs["predicted_doublet"] == False].copy()

    if adata_sample.n_obs > 0:
        scrub_results.append(adata_sample)
        print(f"Sample {sample}: {adata_sample.n_obs} cells remain after doublet removal")
    else:
        print(f"Warning: Sample {sample} has 0 cells after doublet removal. Skipping.")

if scrub_results:
    sample_keys = [adata.obs[SAMPLE_COL].iloc[0] for adata in scrub_results]
    adata = anndata.concat(scrub_results, join="outer", label=SAMPLE_COL, keys=sample_keys)
else:
    raise ValueError("All samples were removed during doublet detection. Please check parameters.")

print("Doublet_Process_finished")

print("n_samples:", adata.obs[SAMPLE_COL].nunique())
print("min cells per sample:", adata.obs[SAMPLE_COL].value_counts().min())

In [ ]:
# 生データを "counts" レイヤーに保存
adata.layers["counts"] = adata.X.copy()

# 正規化と対数変換
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# 正規化前の生データを raw 属性に保存
adata.raw = adata.copy()

# HVG選択→スケーリング
sc.pp.highly_variable_genes(adata, n_top_genes=4000, flavor='seurat_v3', layer='counts')
sc.pl.highly_variable_genes(adata, show=False, save=os.path.join(OUTPUT_DIR, "highly_variable_genes.png"))

# 前処理後のデータを保存
adata.write_h5ad(os.path.join(OUTPUT_DIR, "adata_preprocessed.h5ad"))

## 4. クラスタリング最適化と最終クラスタリング（Clustering.py）

In [ ]:
INPUT_DATA = os.path.join(OUTPUT_DIR, "adata_preprocessed.h5ad")

# データの読み込み
adata = sc.read_h5ad(INPUT_DATA)

In [ ]:
# 4.1, PCs数の最適化 (PCAの最適化)
print("\n[1] Optimizing number of PCs...")
n_pcs_list = [10, 15, 20, 25, 30, 40, 50]
pc_results = []

for n_pcs in n_pcs_list:
    print(f"  Testing {n_pcs} PCs...")

    sc.tl.pca(adata, n_comps=n_pcs)
    # バッチ処理（Harmony）
    sce.pp.harmony_integrate(adata, key=SAMPLE_COL, max_iter_harmony=20)

    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=n_pcs, use_rep='X_pca_harmony')
    sc.tl.leiden(adata, resolution=0.6)

    if adata.n_obs > 5000:
        np.random.seed(42)
        sample_idx = np.random.choice(adata.n_obs, 5000, replace=False)
        X_sample = adata.obsm['X_pca_harmony'][sample_idx, :n_pcs]
        labels_sample = adata.obs['leiden'].iloc[sample_idx].astype(int).values
    else:
        X_sample = adata.obsm['X_pca_harmony'][:, :n_pcs]
        labels_sample = adata.obs['leiden'].astype(int).values

    sil_score = silhouette_score(X_sample, labels_sample)
    db_score = davies_bouldin_score(X_sample, labels_sample)
    ch_score = calinski_harabasz_score(X_sample, labels_sample)
    n_clusters = len(np.unique(labels_sample))

    pc_results.append({
        'n_pcs': n_pcs,
        'silhouette': sil_score,
        'davies_bouldin': db_score,
        'calinski_harabasz': ch_score,
        'n_clusters': n_clusters
    })

    print(f"    {n_pcs} PCs: Silhouette={sil_score:.4f}, DB={db_score:.4f}, Clusters={n_clusters}")

pc_df = pd.DataFrame(pc_results)
pc_df.to_csv(os.path.join(OUTPUT_DIR, 'pcs_results.csv'), index=False)

In [ ]:
# PCs数の最適化の可視化 (Silhouette, Davies-Bouldin, Calinski-Harabasz)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(pc_df['n_pcs'], pc_df['silhouette'], 'o-', color='purple')
axes[0].axvline(pc_df.loc[pc_df['silhouette'].idxmax(), 'n_pcs'], color='red', linestyle='--', alpha=0.7)
axes[0].set_xlabel('Number of PCs')
axes[0].set_ylabel('Silhouette Score')
axes[0].set_title('Silhouette Score vs PCs')
axes[0].grid(True, alpha=0.3)

axes[1].plot(pc_df['n_pcs'], pc_df['davies_bouldin'], 'o-', color='green')
axes[1].axvline(pc_df.loc[pc_df['davies_bouldin'].idxmin(), 'n_pcs'], color='red', linestyle='--', alpha=0.7)
axes[1].set_xlabel('Number of PCs')
axes[1].set_ylabel('Davies-Bouldin Index')
axes[1].set_title('DB Index vs PCs (lower is better)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(pc_df['n_pcs'], pc_df['calinski_harabasz'], 'o-', color='orange')
axes[2].axvline(pc_df.loc[pc_df['calinski_harabasz'].idxmax(), 'n_pcs'], color='red', linestyle='--', alpha=0.7)
axes[2].set_xlabel('Number of PCs')
axes[2].set_ylabel('Calinski-Harabasz Index')
axes[2].set_title('CH Index vs PCs (higher is better)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'pcs_optimization.png'), dpi=300)
plt.close()

best_n_pcs = int(pc_df.loc[pc_df['silhouette'].idxmax(), 'n_pcs'])
print(f"\nBest number of PCs: {best_n_pcs} (Silhouette: {pc_df['silhouette'].max():.4f})")

In [ ]:
# 4.2, k-neighborsの最適化 (Leidenの近傍数の最適化)
print("\n[2] Optimizing k-neighbors...")
sc.tl.pca(adata, n_comps=best_n_pcs)

# バッチ処理（Harmony）
sce.pp.harmony_integrate(adata, key=SAMPLE_COL, max_iter_harmony=20)

k_values = [5, 10, 15, 20, 25, 30, 40, 50]
k_results = []

for k in k_values:
    print(f"  Testing k={k}...")

    sc.pp.neighbors(adata, n_neighbors=k, n_pcs=best_n_pcs, use_rep='X_pca_harmony')
    sc.tl.leiden(adata, resolution=0.6)

    if adata.n_obs > 5000:
        np.random.seed(42)
        sample_idx = np.random.choice(adata.n_obs, 5000, replace=False)
        X_sample = adata.obsm['X_pca_harmony'][sample_idx, :best_n_pcs]
        labels_sample = adata.obs['leiden'].iloc[sample_idx].astype(int).values
    else:
        X_sample = adata.obsm['X_pca_harmony'][:, :best_n_pcs]
        labels_sample = adata.obs['leiden'].astype(int).values

    sil_score = silhouette_score(X_sample, labels_sample)
    db_score = davies_bouldin_score(X_sample, labels_sample)
    ch_score = calinski_harabasz_score(X_sample, labels_sample)
    n_clusters = len(np.unique(labels_sample))

    k_results.append({
        'k': k,
        'silhouette': sil_score,
        'davies_bouldin': db_score,
        'calinski_harabasz': ch_score,
        'n_clusters': n_clusters
    })

    print(f"    k={k}: Silhouette={sil_score:.4f}, DB={db_score:.4f}, Clusters={n_clusters}")

k_df = pd.DataFrame(k_results)
k_df.to_csv(os.path.join(OUTPUT_DIR, 'k_neighbors_results.csv'), index=False)

In [ ]:
# k-neighborsの最適化の可視化 (Silhouette, Davies-Bouldin, Calinski-Harabasz)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(k_df['k'], k_df['silhouette'], 'o-', color='orange')
axes[0].axvline(k_df.loc[k_df['silhouette'].idxmax(), 'k'], color='red', linestyle='--', alpha=0.7)
axes[0].set_xlabel('k-neighbors')
axes[0].set_ylabel('Silhouette Score')
axes[0].set_title('Silhouette Score vs k-neighbors')
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_df['k'], k_df['davies_bouldin'], 'o-', color='green')
axes[1].axvline(k_df.loc[k_df['davies_bouldin'].idxmin(), 'k'], color='red', linestyle='--', alpha=0.7)
axes[1].set_xlabel('k-neighbors')
axes[1].set_ylabel('Davies-Bouldin Index')
axes[1].set_title('DB Index vs k-neighbors')
axes[1].grid(True, alpha=0.3)

axes[2].plot(k_df['k'], k_df['calinski_harabasz'], 'o-', color='blue')
axes[2].axvline(k_df.loc[k_df['calinski_harabasz'].idxmax(), 'k'], color='red', linestyle='--', alpha=0.7)
axes[2].set_xlabel('k-neighbors')
axes[2].set_ylabel('Calinski-Harabasz Index')
axes[2].set_title('CH Index vs k-neighbors')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'k_neighbors_optimization.png'), dpi=300)
plt.close()

best_k = int(k_df.loc[k_df['silhouette'].idxmax(), 'k'])
print(f"\nBest k-neighbors: {best_k} (Silhouette: {k_df['silhouette'].max():.4f})")

In [ ]:
# 4.3, Resolution最適化　(Leidenの解像度の最適化)
print("\n[3] Optimizing resolution...")
sc.pp.neighbors(adata, n_neighbors=best_k, n_pcs=best_n_pcs, use_rep='X_pca_harmony')

resolutions = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0]
resolution_results = []

for res in resolutions:
    print(f"  Testing resolution={res:.2f}...")

    sc.tl.leiden(adata, resolution=res, key_added=f'leiden_res_{res}')

    if adata.n_obs > 5000:
        np.random.seed(42)
        sample_idx = np.random.choice(adata.n_obs, 5000, replace=False)
        X_sample = adata.obsm['X_pca_harmony'][sample_idx, :best_n_pcs]
        labels_sample = adata.obs[f'leiden_res_{res}'].iloc[sample_idx].astype(int).values
    else:
        X_sample = adata.obsm['X_pca_harmony'][:, :best_n_pcs]
        labels_sample = adata.obs[f'leiden_res_{res}'].astype(int).values

    sil_score = silhouette_score(X_sample, labels_sample)
    db_score = davies_bouldin_score(X_sample, labels_sample)
    ch_score = calinski_harabasz_score(X_sample, labels_sample)
    n_clusters = len(np.unique(labels_sample))

    resolution_results.append({
        'resolution': res,
        'silhouette': sil_score,
        'davies_bouldin': db_score,
        'calinski_harabasz': ch_score,
        'n_clusters': n_clusters
    })

    print(f"    Resolution {res:.2f}: Silhouette={sil_score:.4f}, DB={db_score:.4f}, Clusters={n_clusters}")

res_df = pd.DataFrame(resolution_results)
res_df.to_csv(os.path.join(OUTPUT_DIR, 'resolution_results.csv'), index=False)

In [ ]:
# Resolution最適化の可視化 (Silhouette, Davies-Bouldin, Calinski-Harabasz)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes[0, 0].plot(res_df['resolution'], res_df['silhouette'], 'o-', color='blue')
axes[0, 0].axvline(res_df.loc[res_df['silhouette'].idxmax(), 'resolution'], color='red', linestyle='--', alpha=0.7)
axes[0, 0].set_xlabel('Resolution')
axes[0, 0].set_ylabel('Silhouette Score')
axes[0, 0].set_title('Silhouette Score vs Resolution')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(res_df['resolution'], res_df['davies_bouldin'], 'o-', color='green')
axes[0, 1].axvline(res_df.loc[res_df['davies_bouldin'].idxmin(), 'resolution'], color='red', linestyle='--', alpha=0.7)
axes[0, 1].set_xlabel('Resolution')
axes[0, 1].set_ylabel('Davies-Bouldin Index')
axes[0, 1].set_title('DB Index vs Resolution')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(res_df['resolution'], res_df['calinski_harabasz'], 'o-', color='purple')
axes[1, 0].axvline(res_df.loc[res_df['calinski_harabasz'].idxmax(), 'resolution'], color='red', linestyle='--', alpha=0.7)
axes[1, 0].set_xlabel('Resolution')
axes[1, 0].set_ylabel('Calinski-Harabasz Index')
axes[1, 0].set_title('CH Index vs Resolution')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(res_df['resolution'], res_df['n_clusters'], 'o-', color='orange')
axes[1, 1].set_xlabel('Resolution')
axes[1, 1].set_ylabel('Number of Clusters')
axes[1, 1].set_title('Cluster Count vs Resolution')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'resolution_optimization.png'), dpi=300)
plt.close()

best_resolution = res_df.loc[res_df['silhouette'].idxmax(), 'resolution']
print(f"\nBest resolution: {best_resolution:.2f} (Silhouette: {res_df['silhouette'].max():.4f})")

In [ ]:
# 4.4, パラメータ探索のサマリー作成
print("\n=== Optimization Summary ===")
print(f"Best number of PCs: {best_n_pcs} (Silhouette: {pc_df['silhouette'].max():.4f})")
print(f"Best k-neighbors: {best_k} (Silhouette: {k_df['silhouette'].max():.4f})")
print(f"Best resolution: {best_resolution:.2f} (Silhouette: {res_df['silhouette'].max():.4f})")

summary_df = pd.DataFrame({
    'parameter': ['n_pcs', 'k_neighbors', 'resolution'],
    'optimal_value': [best_n_pcs, best_k, best_resolution],
    'max_silhouette': [pc_df['silhouette'].max(),
                      k_df['silhouette'].max(),
                      res_df['silhouette'].max()]
})
summary_df.to_csv(os.path.join(OUTPUT_DIR, 'optimization_summary.csv'), index=False)

In [ ]:
# 4.5, バッチ処理と最適化したパラメータで最終クラスタリング
print("\n[4.5] Final clustering with optimal parameters...")

# PCA
sc.tl.pca(adata, n_comps=best_n_pcs)
# バッチ処理（Harmony）
sce.pp.harmony_integrate(adata, key=SAMPLE_COL, max_iter_harmony=20)
# 最適化したパラメータで近傍グラフ構築とクラスタリング
sc.pp.neighbors(adata, n_neighbors=best_k, n_pcs=best_n_pcs, use_rep='X_pca_harmony')
# 最適化した解像度でLeidenクラスタリング
sc.tl.leiden(adata, resolution=best_resolution)
# UMAPの計算
sc.tl.umap(adata)

# クラスタリング結果の保存
adata.write_h5ad(os.path.join(OUTPUT_DIR, 'adata_clustered.h5ad'))

## 5. クラスタリング可視化（Clustering_Visualization.py）

In [ ]:
INPUT_DATA = os.path.join(OUTPUT_DIR, "adata_clustered.h5ad")

# データの読み込み
adata = sc.read_h5ad(INPUT_DATA)

# クラスタリング結果のUMAPプロット
sc.pl.umap(adata, color='leiden', save='_leiden_clusters.png')
# サンプルごとのUMAPプロット
sc.pl.umap(adata, color=SAMPLE_COL, save=f'_{SAMPLE_COL}.png')
# ミトコンドリア遺伝子の割合をUMAP上で可視化
sc.pl.umap(adata, color='pct_counts_mt', save='_pct_counts_mt.png')

## 6. CellTypist自動アノテーション（Celltypist _Annotation.py）

In [ ]:
INPUT_DATA = os.path.join(OUTPUT_DIR, "adata_clustered.h5ad")

# データの読み込み
adata = sc.read_h5ad(INPUT_DATA)

# CellTypistのモデルをダウンロードして読み込む
models.download_models(model=MODEL_NAME, force_update=True)
model = models.Model.load(model=MODEL_NAME)
predictions = celltypist.annotate(
    adata,
    model=model,
    majority_voting=True
)
adata = predictions.to_adata()

print("Celltypist_Process_finished")

adata.write_h5ad(os.path.join(OUTPUT_DIR, 'adata_annotated_celltypist.h5ad'))

## 7. マーカー遺伝子によるアノテーション（Marker_Annotation.py）

In [ ]:
INPUT_DATA = os.path.join(OUTPUT_DIR, "adata_annotated_celltypist.h5ad")

# データの読み込み
adata = sc.read_h5ad(INPUT_DATA)

# マーカー遺伝子のリストの読み込み
with open(MARKER_FILE, "r", encoding="utf-8") as marker_handle:
    markers_dict = ast.literal_eval(marker_handle.read())

all_genes = []
seen_genes = set()
for gene_list in markers_dict.values():
    for gene in gene_list:
        if gene not in seen_genes:
            seen_genes.add(gene)
            all_genes.append(gene)

In [ ]:
# クラスターごとにマーカー遺伝子の発現をドットプロットで確認
sc.pl.dotplot(adata, var_names=all_genes, groupby='leiden', save='_marker_dotplot.png')
# クラスターごとにマーカー遺伝子の発現をヒートマップで確認
sc.pl.heatmap(adata, var_names=all_genes, groupby='leiden', save='_marker_heatmap.png')
# クラスターごとにマーカー遺伝子の発現をバイオリンプロットで確認
sc.pl.violin(adata, var_names=all_genes, groupby='leiden', save='_marker_violin.png')
#UMAP上でマーカー遺伝子の発現を可視化
for gene in all_genes:
    sc.pl.umap(adata, color=gene, save='_' + gene + '_umap.png')

In [ ]:
# Module Scoreの計算
for cluster, gene_list in markers_dict.items():
    adata.obs[cluster + '_score'] = sc.tl.score_genes(
        adata,
        gene_list=gene_list,
        score_name=cluster + '_score',
        use_raw=False,
    )
# Module Scoreのドットプロット
score_cols = [col for col in adata.obs.columns if col.endswith('_score')]
sc.pl.dotplot(adata, var_names=score_cols, groupby='leiden',
                save='_module_score_dotplot.png')
# Module Scoreのヒートマップ
sc.pl.heatmap(adata, var_names=score_cols, groupby='leiden',
                save='_module_score_heatmap.png')
# Module Scoreのバイオリンプロット
sc.pl.violin(adata, var_names=score_cols, groupby='leiden',
                save='_module_score_violin.png')

# UMAP上でModule Scoreを可視化
for score in score_cols:
    sc.pl.umap(adata, color=score, save='_' + score + '_umap.png')

adata.write_h5ad(os.path.join(OUTPUT_DIR, 'adata_marker_annotated.h5ad'))

## 8. クラスターID→細胞型マッピング（Annotation_detct.py）

In [ ]:
INPUT_DATA = os.path.join(OUTPUT_DIR, "adata_marker_annotated.h5ad")

# データの読み込み
adata = sc.read_h5ad(INPUT_DATA)

# クラスターごとにアノテーションをマッピング
with open(ANNOTATION_FILE, "r", encoding="utf-8") as annotation_handle:
    annotation_map = json.load(annotation_handle)

# クラスターIDを文字列に変換してアノテーションをマッピング
adata.obs['leiden'] = adata.obs['leiden'].astype(str)
adata.obs['cell_type'] = adata.obs['leiden'].map(annotation_map)

In [ ]:
# UMAP上でアノテーションを可視化
sc.pl.umap(adata, color='cell_type', save='_annotation_umap.png')
# Cell typeごとのマーカー遺伝子の発現をドットプロットで確認
sc.pl.dotplot(adata, var_names=adata.var_names, groupby='cell_type', save='_annotation_dotplot.png')
# Cell typeごとのマーカー遺伝子の発現をヒートマップで確認
sc.pl.heatmap(adata, var_names=adata.var_names, groupby='cell_type', save='_annotation_heatmap.png')
# Cell typeごとのマーカー遺伝子の発現をバイオリンプロットで確認
sc.pl.violin(adata, var_names=adata.var_names, groupby='cell_type', save='_annotation_violin.png')

# データの保存
adata.write_h5ad(os.path.join(OUTPUT_DIR, 'annotated_data.h5ad'))

## 9. 各ステップの出力ファイル確認

In [ ]:
patterns = ["*.h5ad", "*.png", "*.csv", "*.json"]
files = []
for pattern in patterns:
    files.extend(glob.glob(os.path.join(OUTPUT_DIR, pattern)))

for path in sorted(files):
    print(path)